# Whisper large-v3 LoRA hyperparameter sweep: Greek council ASR

Sweeps 9 training configs (3 LR x 3 LoRA-rank) on the fixed data recipe (corrections +
1x no-edit backbone) and writes a regression-guarded leaderboard. Epochs are read off the
per-epoch eval curve, not run separately. No LLM / API key is involved: Kaggle only trains
and emits metrics.

Run on Kaggle: Accelerator = GPU T4 x2, Internet On, then Save & Run All (Commit).
Smoke first (`SMOKE = True`) to prove the harness, then `SMOKE = False` for the ~3-4h run.

Memory note (carried from the single-config notebook): each meeting's mp3 is decoded to
PCM one at a time and freed before the next. An all-in-RAM version OOM'd.

Spec: docs/specs/whisper-hyperparam-sweep.md.

In [ ]:
# 1. Install
# NB: Kaggle ships torchao==0.10.0, which PEFT's LoRA dispatcher rejects with an
# ImportError ("only versions above 0.16.0 are supported"). We don't use torchao
# (plain fp16 LoRA), and is_torchao_available() returns False cleanly when it's
# absent; so just remove it.
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install -U transformers datasets peft accelerate evaluate jiwer faster-whisper librosa soundfile 2>/dev/null
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Sweep config
EXPORT_URL = "https://79-76-114-184.sslip.io/api/export"
MEETING_API = "https://opencouncil.gr/api/cities/{city}/meetings/{meeting}"
MODEL_ID   = "openai/whisper-large-v3"
LANGUAGE, TASK = "greek", "transcribe"
VAL_CITIES = {"orestiada", "argos"}
SMOKE      = True            # True = tiny sanity sweep; False = full ~3-4h Standard run
SEED       = 13
# Search space
LRS   = [5e-5, 1e-4, 2e-4]
RANKS = [8, 16, 32]
EPOCHS = 4
TRAIN_BS, GRAD_ACC, EVAL_BS = 2, 4, 4
LORA_DROPOUT = 0.05
VAL_SUBSAMPLE = 70           # clips per val set kept for eval (keeps per-epoch eval bounded)
MAX_REG_DELTA = 1.0          # best-pick guard: skip configs whose val_reg regresses > this
# Data-build sizing (mirror reference notebook; SMOKE caps distinct meetings)
SAMPLE_N = 60 if SMOKE else None
SMOKE_TRAIN_MEETINGS = 4 if SMOKE else None
SMOKE_VAL_MEETINGS   = 2 if SMOKE else None
VAL_REG_PER_MEETING = 8
SR = 16000; PAD_S = 0.2; MIN_DUR, MAX_DUR = 0.3, 30.0
OUT_DIR = "/kaggle/working/whisper-sweep"
import os, random, numpy as np
os.makedirs(OUT_DIR, exist_ok=True); random.seed(SEED); np.random.seed(SEED)

In [ ]:
# 2b. Sweep helpers (inlined from eval/sweep/sweep_utils.py -- Kaggle kernels are
# self-contained). Keep in sync with that module; it is the tested source of truth.
import math, random as _rnd
def _finite(x):
    return isinstance(x, (int, float)) and math.isfinite(x)
def _disp(v):  # round floats to 3dp for display only
    return round(v, 3) if isinstance(v, float) else v
def make_grid(lrs, ranks, seed):
    grid = []
    for lr in lrs:
        for rank in ranks:
            grid.append({"config_id": f"lr{lr:g}_r{rank}", "lr": lr, "rank": rank,
                         "alpha": 2 * rank, "seed": seed})
    return grid
def _subsample_idx(n_total, n, seed):
    if n is None or n_total <= n:
        return list(range(n_total))
    r = _rnd.Random(seed)
    return sorted(r.sample(range(n_total), n))
def build_leaderboard(rows, baseline):
    base = baseline.get("val_reg_wer")
    out = []
    for r in rows:
        rr = dict(r); v = r.get("val_reg_wer")
        rr["reg_delta"] = (v - base) if (_finite(v) and _finite(base)) else None
        out.append(rr)
    out.sort(key=lambda x: x["val_corr_wer_norm"]); return out
def pick_best(sorted_rows, max_reg_delta=1.0):
    for r in sorted_rows:
        d = r.get("reg_delta")
        if d is None or d <= max_reg_delta:
            return r
    return None
_COLS = ["config_id","lr","rank","alpha","epoch","val_corr_wer_norm","val_reg_wer",
         "reg_delta","val_corr_cer","eval_loss","wall_s"]
def render_markdown(sorted_rows, best):
    lines = ["| " + " | ".join(_COLS) + " |", "| " + " | ".join("---" for _ in _COLS) + " |"]
    for r in sorted_rows:
        lines.append("| " + " | ".join(str(_disp(r.get(c, ""))) for c in _COLS) + " |")
    bl = (f"**Best (regression-guarded):** {best['config_id']} (epoch {best['epoch']}, "
          f"val_corr_wer_norm {best['val_corr_wer_norm']}, reg_delta {_disp(best['reg_delta'])})"
          if best else "**Best (regression-guarded):** none -- every config regressed val_reg")
    return "\n".join(lines) + "\n\n" + bl + "\n"
print("grid:", [c["config_id"] for c in make_grid(LRS, RANKS, SEED)])

In [ ]:
# 3. Fetch export + meeting JSON helper
import requests, json, collections
def fetch_jsonl(url):
    r = requests.get(url, timeout=600); r.raise_for_status()
    return [json.loads(l) for l in r.text.splitlines() if l.strip()]
rows = fetch_jsonl(EXPORT_URL)
print("included rows:", len(rows))
print("per city:", dict(collections.Counter(r["city_id"] for r in rows).most_common()))
def fetch_meeting(city, meeting):
    r = requests.get(MEETING_API.format(city=city, meeting=meeting),
                     headers={"User-Agent":"oc-ft/1.0","Accept":"application/json"}, timeout=120)
    r.raise_for_status(); return r.json()

# Drop denylisted (unreviewed, <5% human-edit) meetings -- defense in depth in
# case the export wasn't rebuilt yet. Canonical list: data/exclusions/ in the repo.
try:
    _exj = requests.get("https://raw.githubusercontent.com/eellak/gsoc2026-opencouncil-stt/main/data/exclusions/unreviewed_meetings.json", timeout=60).json()
    _excl = {(m["city_id"], m["meeting_id"]) for m in _exj.get("meetings", [])}
    _b = len(rows); rows = [r for r in rows if (r["city_id"], r["meeting_id"]) not in _excl]
    print(f"denylist: dropped {_b-len(rows)} rows from {len(_excl)} excluded meetings")
except Exception as e:
    print("denylist filter skipped:", e)

In [ ]:
# 4. Audio helpers (NO global PCM cache; decode per meeting, free after)
import librosa, hashlib, pathlib, gc
CACHE = pathlib.Path("/tmp/audio_cache"); CACHE.mkdir(parents=True, exist_ok=True)
def dl(url):
    # Write through a temp file and atomically move it into place, so a failed/partial
    # download is never cached and later reused as if it were complete.
    p = CACHE / (hashlib.md5(url.encode()).hexdigest() + ".mp3")
    if not p.exists():
        tmp = p.with_suffix(p.suffix + ".tmp")
        try:
            with requests.get(url, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for c in r.iter_content(1 << 20):
                        if c: f.write(c)
            os.replace(tmp, p)
        finally:
            if tmp.exists(): tmp.unlink()
    return str(p)
def load_pcm_path(path): return librosa.load(path, sr=SR, mono=True)[0]
def cut(y, start, end):
    a = max(0, int((start - PAD_S) * SR)); b = min(len(y), int((end + PAD_S) * SR)); return y[a:b]
def ok_span(s, e):
    if s is None or e is None: return False   # reject missing timestamps (would break cut())
    return MIN_DUR <= (e - s) <= MAX_DUR

In [ ]:
# 5. Memory-safe build with CACHE GUARD: persist clips+manifest to /kaggle/working so a
#    kernel restart does NOT re-decode every meeting (~70 min). Rebuild only if the signature changes.
import gc, json, hashlib, soundfile as sf
WORK = pathlib.Path("/kaggle/working"); CLIPS = WORK / "clips"; CLIPS.mkdir(parents=True, exist_ok=True)
MAN_PATH = WORK / "manifest.json"

# Signature = everything that changes the built clips. Stale cache (mismatch) -> rebuild.
# SEED is included: it drives meeting sampling, so changing it must invalidate the cache.
_sig = {"ver": 3, "sr": SR, "seed": SEED, "pad_s": PAD_S, "min_dur": MIN_DUR, "max_dur": MAX_DUR,
        "val_cities": sorted(VAL_CITIES),
        "smoke_train": SMOKE_TRAIN_MEETINGS, "smoke_val": SMOKE_VAL_MEETINGS,
        "sample_n": SAMPLE_N, "val_reg_per_mtg": VAL_REG_PER_MEETING, "n_included": len(rows)}
_sig_str = json.dumps(_sig, sort_keys=True, ensure_ascii=False)

man = None
if MAN_PATH.exists():
    _c = json.load(open(MAN_PATH))
    _all = [c["audio"] for s in ("train","valc","valr") for c in _c.get(s, [])]  # check every clip
    if _c.get("_sig") == _sig_str and _all and all(pathlib.Path(a).is_file() for a in _all):
        man = {k: _c[k] for k in ("train","valc","valr")}
        print(f"CACHE HIT -> train={len(man['train'])} valc={len(man['valc'])} valr={len(man['valr'])} (skipped ~70min rebuild)")
    else:
        print("cache present but signature/clips mismatch -> rebuilding")

if man is None:
    train_src = [r for r in rows if r["city_id"] not in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]
    val_src   = [r for r in rows if r["city_id"] in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]

    def cap_meetings(src, n):
        # Keep only rows from n randomly-chosen meetings -> fewer full-mp3 downloads.
        # sorted() before shuffle makes the pick reproducible: a plain set has no stable
        # iteration order, so the same SEED could otherwise pick different meetings.
        if not n: return src
        mids = sorted({(r["city_id"], r["meeting_id"]) for r in src})
        random.Random(SEED).shuffle(mids); keep = set(mids[:n])
        return [r for r in src if (r["city_id"], r["meeting_id"]) in keep]

    train_src = cap_meetings(train_src, SMOKE_TRAIN_MEETINGS)
    val_src   = cap_meetings(val_src, SMOKE_VAL_MEETINGS)
    if SAMPLE_N: random.shuffle(train_src); train_src = train_src[:SAMPLE_N]
    print(f"sources: train_src={len(train_src)} ({len({(r['city_id'],r['meeting_id']) for r in train_src})} mtgs) "
          f"val_src={len(val_src)} ({len({(r['city_id'],r['meeting_id']) for r in val_src})} mtgs)")

    reg_src = []
    for city, mtg in sorted({(r["city_id"], r["meeting_id"]) for r in val_src}):
        try: mj = fetch_meeting(city, mtg)
        except Exception as e: print("skip", city, mtg, e); continue
        au = (mj.get("meeting") or {}).get("audioUrl")
        if not au: continue
        ne = [u for seg in (mj.get("transcript") or []) for u in (seg.get("utterances") or [])
              if u.get("lastModifiedBy") is None and ok_span(u.get("startTimestamp"), u.get("endTimestamp")) and (u.get("text") or "").strip()]
        random.shuffle(ne)
        for u in ne[:VAL_REG_PER_MEETING]:
            reg_src.append({"audio_url": au, "start": u["startTimestamp"], "end": u["endTimestamp"], "text": u["text"]})

    def mk(r, dst, k): return {"url": r["audio_url"], "start": r["start"], "end": r["end"], "text": r[k], "dst": dst}
    tasks = ([mk(r,"train","final_after_text") for r in train_src]
           + [mk(r,"valc","final_after_text") for r in val_src]
           + [mk(r,"valr","text") for r in reg_src])
    buckets = collections.defaultdict(list)
    for t in tasks: buckets[t["url"]].append(t)

    man = {"train": [], "valc": [], "valr": []}
    fail = 0; idx = 0
    for i, (url, ts) in enumerate(buckets.items(), 1):
        try:
            path = dl(url); y = load_pcm_path(path)        # ONE meeting in RAM at a time
        except Exception as e:
            fail += len(ts); print("audio fail", url, str(e)[:60]); continue
        for t in ts:
            wav = str(CLIPS / f"{t['dst']}_{idx}.wav"); idx += 1
            sf.write(wav, cut(y, t["start"], t["end"]), SR)   # stream clip to disk
            man[t["dst"]].append({"audio": wav, "text": (t["text"] or "").strip()})
        del y; gc.collect()                                # free RAM + disk before next meeting
        try: os.remove(path)
        except Exception: pass
        if i % 20 == 0: print(f"{i}/{len(buckets)} meetings | train={len(man['train'])} valc={len(man['valc'])} valr={len(man['valr'])}")
    print(f"DONE train={len(man['train'])} valc={len(man['valc'])} valr={len(man['valr'])} audio_fail={fail}")
    json.dump({**man, "_sig": _sig_str}, open(MAN_PATH, "w"), ensure_ascii=False)
    print("cached manifest ->", MAN_PATH)

In [ ]:
# 6. HF datasets (LAZY audio from disk) + Whisper preprocessing -> Arrow on disk
from datasets import Dataset, Audio
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
def to_ds(recs):
    if not recs: return None
    d = Dataset.from_list(recs).cast_column("audio", Audio(sampling_rate=SR))
    def prep(b):
        a = b["audio"]
        b["input_features"] = processor.feature_extractor(a["array"], sampling_rate=a["sampling_rate"]).input_features[0]
        b["labels"] = processor.tokenizer(b["text"]).input_ids
        return b
    return d.map(prep, remove_columns=["audio","text"])
ds_train = to_ds(man["train"]); ds_valc = to_ds(man["valc"]); ds_valr = to_ds(man["valr"])
gc.collect()
# Keep per-epoch eval bounded: deterministically subsample the val sets (same every run).
ds_valc = ds_valc.select(_subsample_idx(ds_valc.num_rows, VAL_SUBSAMPLE, SEED))
if ds_valr: ds_valr = ds_valr.select(_subsample_idx(ds_valr.num_rows, VAL_SUBSAMPLE, SEED))
print("datasets:", ds_train.num_rows, ds_valc.num_rows, (ds_valr.num_rows if ds_valr else 0))

In [ ]:
# 7. Collator + metrics (raw WER, Greek-normalized WER, CER)
# NOTE: decode with clean_up_tokenization_spaces=False. The default (True) is destructive for the
# BPE WhisperTokenizer (strips spaces before punctuation), which corrupts the text and distorts
# WER/CER on punctuation/casing. We also collapse whitespace IDENTICALLY on preds+refs so the
# metric stays fair, and print a few decoded pairs to eyeball the metric inputs.
import torch, evaluate, unicodedata, re
from dataclasses import dataclass
@dataclass
class Collator:
    processor: object
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        # Feature extractor emits float32, but the model is loaded in fp16; match dtypes
        # at the batch boundary so encoder conv1 doesn't crash in generate() during eval
        # ("Input type (float) and bias type (c10::Half) should be the same").
        batch["input_features"] = batch["input_features"].to(torch.float16)
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:,0] == self.processor.tokenizer.bos_token_id).all().cpu().item(): labels = labels[:,1:]
        batch["labels"] = labels; return batch
collator = Collator(processor)
wer_m, cer_m = evaluate.load("wer"), evaluate.load("cer")
_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
def _ws(s):  # strip + collapse internal whitespace; applied to BOTH sides so WER stays fair
    return " ".join(s.split())
def gnorm(s):
    s = unicodedata.normalize("NFD", s.lower()); s = "".join(c for c in s if unicodedata.category(c)!="Mn")
    s = unicodedata.normalize("NFC", s).replace("ς","σ"); return re.sub(r"\s+"," ", _PUNCT.sub(" ", s)).strip()
_DBG = {"n": 0}
def metrics(pred):
    lab = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)
    P = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    R = processor.tokenizer.batch_decode(lab,             skip_special_tokens=True, clean_up_tokenization_spaces=False)
    if _DBG["n"] < 2:  # eyeball metric inputs once or twice after the decode change
        for p, r in list(zip(P, R))[:3]: print("PRED:", repr(p)); print("REF :", repr(r))
        _DBG["n"] += 1
    Pw, Rw = [_ws(x) for x in P], [_ws(x) for x in R]
    return {"wer": 100*wer_m.compute(predictions=Pw, references=Rw),
            "wer_norm": 100*wer_m.compute(predictions=[gnorm(x) for x in P], references=[gnorm(x) for x in R]),
            "cer": 100*cer_m.compute(predictions=[x.strip() for x in P], references=[x.strip() for x in R])}

In [ ]:
# 8. Sweep loop: 9 configs (3 LR x 3 LoRA-rank), epochs read off the per-epoch curve.
# Reload the fp16 base fresh per config for clean isolation across ranks; a callback
# evals val_reg after each epoch so the regression guard is per-epoch.
import time, gc
from transformers import (WhisperForConditionalGeneration, Seq2SeqTrainingArguments,
                          Seq2SeqTrainer, TrainerCallback)
from peft import LoraConfig, get_peft_model

class RegEvalCallback(TrainerCallback):
    # After each epoch, eval val_reg and stash its wer into the sink dict by epoch.
    def __init__(self, trainer, ds_valr, sink):
        self.trainer, self.ds_valr, self.sink = trainer, ds_valr, sink
    def on_epoch_end(self, args, state, control, **kw):
        if self.ds_valr is None: return
        m = self.trainer.evaluate(self.ds_valr, metric_key_prefix="valr")
        self.sink[round(state.epoch)] = m.get("valr_wer", float("nan"))

def build_model(rank, alpha):
    m = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    m.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
    m.config.suppress_tokens = []
    m.generation_config.language, m.generation_config.task = LANGUAGE, TASK
    m.model.encoder.requires_grad_(False)
    m.gradient_checkpointing_enable(); m.config.use_cache = False
    m = get_peft_model(m, LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
                                     target_modules=["q_proj","v_proj"]))
    return m

# Baseline (epoch 0) once, on the untouched base model.
_base = build_model(8, 16)
_bargs = Seq2SeqTrainingArguments(output_dir=OUT_DIR+"/_base", per_device_eval_batch_size=EVAL_BS,
    predict_with_generate=True, generation_max_length=225, fp16=True, report_to=[],
    remove_unused_columns=False, label_names=["labels"])
_bt = Seq2SeqTrainer(model=_base, args=_bargs, data_collator=collator,
                     compute_metrics=metrics, processing_class=processor)
baseline = {"val_corr_wer_norm": round(_bt.evaluate(ds_valc)["eval_wer_norm"], 3),
            "val_reg_wer": (round(_bt.evaluate(ds_valr)["eval_wer"], 3) if ds_valr else float("nan"))}
print("BASELINE", baseline)
del _base, _bt; gc.collect(); torch.cuda.empty_cache()

rows = []
for cfg in make_grid(LRS, RANKS, SEED):
    t0 = time.time()
    model = build_model(cfg["rank"], cfg["alpha"])
    args = Seq2SeqTrainingArguments(output_dir=f"{OUT_DIR}/{cfg['config_id']}",
        per_device_train_batch_size=TRAIN_BS, gradient_accumulation_steps=GRAD_ACC,
        learning_rate=cfg["lr"], warmup_ratio=0.1, num_train_epochs=EPOCHS, fp16=True,
        predict_with_generate=True, generation_max_length=225, eval_strategy="epoch",
        save_strategy="no", logging_steps=20, report_to=[], remove_unused_columns=False,
        label_names=["labels"], seed=cfg["seed"], per_device_eval_batch_size=EVAL_BS)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds_train,
        eval_dataset=ds_valc, data_collator=collator, compute_metrics=metrics,
        processing_class=processor)
    reg_by_epoch = {}
    trainer.add_callback(RegEvalCallback(trainer, ds_valr, reg_by_epoch))
    trainer.train()
    for h in trainer.state.log_history:        # one row per epoch from the eval entries
        if "eval_wer_norm" not in h: continue
        ep = round(h["epoch"])
        rows.append({**cfg, "epoch": ep,
            "val_corr_wer_norm": round(h["eval_wer_norm"], 3),
            "val_reg_wer": round(reg_by_epoch.get(ep, float("nan")), 3),
            "val_corr_cer": round(h.get("eval_cer", float("nan")), 3),
            "eval_loss": round(h.get("eval_loss", float("nan")), 4),
            "wall_s": int(time.time() - t0)})
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    print(f"done {cfg['config_id']} in {int(time.time()-t0)}s")

In [ ]:
# 9. Write the leaderboard (CSV + Markdown) to /kaggle/working (downloadable output).
import csv, json
lb = build_leaderboard(rows, baseline)
best = pick_best(lb, max_reg_delta=MAX_REG_DELTA)
with open(OUT_DIR+"/leaderboard.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLS); w.writeheader()
    for r in lb: w.writerow({c: _disp(r.get(c, "")) for c in _COLS})
with open(OUT_DIR+"/leaderboard.md", "w") as f:
    f.write(f"# Sweep leaderboard\n\nbaseline: {json.dumps(baseline)}\n\n")
    f.write(render_markdown(lb, best))
print(render_markdown(lb, best))
print("wrote", OUT_DIR+"/leaderboard.csv")

## Reading the leaderboard

- Sort key is `val_corr_wer_norm` (lower better). `reg_delta` is the change in `val_reg`
  WER vs the pre-train baseline; positive means ordinary speech got worse.
- The **Best** line already excludes configs that regress `val_reg` beyond `MAX_REG_DELTA`.
- Single seed + small sample: prior CPU research showed seed-variance exceeded
  config-ranking differences. Use the board to reject bad LR/rank, not to crown a 0.1
  winner. Multiple seeds belong to the later, larger run.